# BARAM 2026 — SCADA 정제 미세화 + 블렌드 가중 재스캔 (v8 후보)

v7(LB 0.63497, 139위) 이후. 현행 설정: stuck 임계 30%·가중 0.5·블렌드 w 0.7.

- **1단계**: 임계 0.3 고정, stuck 가중 {0.35, 0.5, 0.65} — GBM·MLP 예측을 따로 저장해 **블렌드 w {0.6,0.7,0.8} 스캔은 무료**
- **2단계**: 1단계 최적 가중에서 임계 {0.2, 0.5} 확인
- 판정: raw blend의 2폴드 공식 총점, **두 해 모두 현행(임계0.3·가중0.5·w0.7) 이상**만 채택
- 캘리브레이션: LB ≈ holdout ±0.002 (v6·v7 실측 확립) — 여기 2024 폴드 개선폭 ≈ LB 개선폭

## 0. 설정 (v7과 동일 부품)

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
GBM_PARAMS=dict(objective="mae",n_estimators=2000,learning_rate=0.020651822836313095,
    num_leaves=63,min_child_samples=300,subsample=0.8,subsample_freq=1,colsample_bytree=0.5,
    reg_lambda=0.1,random_state=42,n_jobs=1,verbose=-1)
RAW=os.environ.get("WIND_RAW_DIR",os.path.expanduser("~/Downloads/open"))
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

# SCADA stuck 비율 테이블 (임계는 나중에 적용할 수 있게 '비율' 자체를 보관)
def stuck_frac():
    frames={}
    for fn,pre,n,rate in [("scada_vestas_train.csv","vestas",12,3600.0),
                          ("scada_unison_train.csv","unison",5,4200.0)]:
        d=pd.read_csv(f"{RAW}/train/{fn}",encoding="utf-8-sig",parse_dates=["kst_dtm"])
        d["hour"]=d["kst_dtm"].dt.ceil("h")
        agg={}
        for i in range(1,n+1):
            pw=f"{pre}_wtg{i:02d}_power_kw10m"; ws=f"{pre}_wtg{i:02d}_ws"
            h=d.groupby("hour").agg(pw_m=(pw,"mean"),ws_m=(ws,"mean"))
            agg[i]=pd.DataFrame({f"stuck_{i}":((h.ws_m>=4.0)&(h.pw_m<=0.01*rate)).astype(float),
                                 f"rep_{i}":h.pw_m.notna().astype(float)})
        frames[pre]=pd.concat(agg.values(),axis=1)
    def frac(pre,ids):
        f=frames[pre]
        st=f[[f"stuck_{i}" for i in ids]].sum(axis=1); rp=f[[f"rep_{i}" for i in ids]].sum(axis=1)
        return (st/rp).where(rp>=3)
    return {1:frac("vestas",range(1,7)),2:frac("vestas",range(7,13)),3:frac("unison",range(1,6))}
FRAC=stuck_frac()
for g in GROUPS:
    s=FRAC[g].reindex(FR[g].kst_dtm).to_numpy()
    FR[g]=FR[g].assign(stuck_frac=np.nan_to_num(s,nan=0.0))

class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_one(pool_tr,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    wt=pool_tr["w"].to_numpy(np.float32)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV)
    gt=torch.tensor(gid,device=DEV); wtt=torch.tensor(wt,device=DEV)
    net=MLP(len(FEATS),**{k:MLP_CFG[k] for k in ("h","depth","drop","emb")}).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=MLP_CFG["lr"],weight_decay=MLP_CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),MLP_CFG["bs"]):
            b=torch.tensor(perm[i:i+MLP_CFG["bs"]],device=DEV); opt.zero_grad()
            e=(net(Xt[b],gt[b])-yt[b]).abs()
            ((e*wtt[b]).sum()/(wtt[b].sum()+1e-8)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            e=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs()
            vl=((e*wtt[cut:]).sum()/(wtt[cut:].sum()+1e-8)).item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

FOLDS={2023:[2022],2024:[2022,2023]}
CACHE={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
    CACHE[vy]=ent
print("folds ready")

folds ready


## 1. 조합 평가기 — GBM·MLP 예측 분리 저장, 블렌드 w 무료 스캔

In [2]:
BLENDS=[0.6,0.7,0.8]
def eval_combo(theta, wgt):
    """(임계 theta, stuck 가중 wgt) → {(vy,B): total}."""
    parts={}
    for vy,ent in CACHE.items():
        # MLP (pooled)
        pool=[]
        for g,(tr2,_) in ent.items():
            p=tr2[FEATS+["kst_dtm","stuck_frac"]].copy()
            p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1
            p["w"]=np.where(p["stuck_frac"]>=theta,wgt,1.0); pool.append(p)
        pool=pd.concat(pool,ignore_index=True)
        mlp={g:[] for g in ent}
        for sd_ in SEEDS:
            net,scaler=train_one(pool,sd_)
            for g,(_,va2) in ent.items():
                mlp[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
        mlp={g:np.mean(v,axis=0) for g,v in mlp.items()}
        gbm={}
        for g,(tr2,va2) in ent.items():
            cap=W.CAP[g]; tgt=TGT[g]
            w=np.where(tr2["stuck_frac"]>=theta,wgt,1.0)
            lg_=lgb.LGBMRegressor(**GBM_PARAMS).fit(tr2[FEATS],tr2[tgt],sample_weight=w)
            hg_=HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
                l2_regularization=1.0,random_state=42).fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy(),sample_weight=w)
            gbm[g]=np.clip(0.5*(lg_.predict(va2[FEATS])+hg_.predict(va2[FEATS].to_numpy())),0,cap)
        for B in BLENDS:
            nm=[]; fi=[]
            for g,(_,va2) in ent.items():
                cap=W.CAP[g]
                p=np.clip((1-B)*gbm[g]+B*mlp[g],0,cap)
                a,b=group_scores(va2[TGT[g]].to_numpy(),p,cap); nm.append(a); fi.append(b)
            parts[(vy,B)]=0.5*(1-np.mean(nm))+0.5*np.mean(fi)
    return parts

RES={}
# 현행 기준선 + 1단계: 가중 스캔 (임계 0.3)
for wgt in [0.35,0.5,0.65]:
    RES[(0.3,wgt)]=eval_combo(0.3,wgt)
    r=RES[(0.3,wgt)]
    print(f"θ=0.3 w={wgt}: " + "  ".join(f"B{B}: 23={r[(2023,B)]:.4f}/24={r[(2024,B)]:.4f}" for B in BLENDS))

θ=0.3 w=0.35: B0.6: 23=0.6075/24=0.6181  B0.7: 23=0.6077/24=0.6170  B0.8: 23=0.6069/24=0.6175


θ=0.3 w=0.5: B0.6: 23=0.6091/24=0.6173  B0.7: 23=0.6098/24=0.6176  B0.8: 23=0.6099/24=0.6172


θ=0.3 w=0.65: B0.6: 23=0.6033/24=0.6144  B0.7: 23=0.6038/24=0.6137  B0.8: 23=0.6037/24=0.6140


In [3]:
# 2단계: 1단계 최적 가중으로 임계 스캔
def best_of(res):
    best=None
    for (th,wg),r in res.items():
        for B in BLENDS:
            key=(th,wg,B); mn=min(r[(2023,B)],r[(2024,B)])
            if best is None or mn>best[1]: best=(key,mn)
    return best
(bth,bwg,bB),_=best_of(RES)
print(f"1단계 최적: θ={bth} w={bwg} B={bB}")
for th in [0.2,0.5]:
    RES[(th,bwg)]=eval_combo(th,bwg)
    r=RES[(th,bwg)]
    print(f"θ={th} w={bwg}: " + "  ".join(f"B{B}: 23={r[(2023,B)]:.4f}/24={r[(2024,B)]:.4f}" for B in BLENDS))

1단계 최적: θ=0.3 w=0.5 B=0.8


θ=0.2 w=0.5: B0.6: 23=0.6091/24=0.6164  B0.7: 23=0.6098/24=0.6165  B0.8: 23=0.6099/24=0.6160


θ=0.5 w=0.5: B0.6: 23=0.6028/24=0.6142  B0.7: 23=0.6020/24=0.6138  B0.8: 23=0.6021/24=0.6131


## 2. 판정

In [4]:
cur=RES[(0.3,0.5)]; cur23,cur24=cur[(2023,0.7)],cur[(2024,0.7)]
print(f"현행(θ0.3·w0.5·B0.7): 2023={cur23:.4f}  2024={cur24:.4f}")
cands=[]
for (th,wg),r in RES.items():
    for B in BLENDS:
        t23,t24=r[(2023,B)],r[(2024,B)]
        if t23>=cur23 and t24>=cur24 and (th,wg,B)!=(0.3,0.5,0.7):
            cands.append(((th,wg,B),min(t23,t24),t23,t24))
if cands:
    (th,wg,B),mn,t23,t24=max(cands,key=lambda x:x[1])
    print(f"채택: θ={th} w={wg} B={B} → 2023={t23:.4f} 2024={t24:.4f} (Δ현행 {t23-cur23:+.4f}/{t24-cur24:+.4f})")
    winner=dict(theta=th,stuck_w=wg,blend=B)
else:
    print("현행을 두 해 모두 이기는 조합 없음 → v7 유지")
    winner=dict(theta=0.3,stuck_w=0.5,blend=0.7)
summary=dict(current={"2023":round(cur23,4),"2024":round(cur24,4)},
    all={f"th{th}_w{wg}_B{B}":{"2023":round(r[(2023,B)],4),"2024":round(r[(2024,B)],4)}
         for (th,wg),r in RES.items() for B in BLENDS},
    winner=winner)
json.dump(summary,open("submission/ver_7/tune2_result.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(winner,ensure_ascii=False))

현행(θ0.3·w0.5·B0.7): 2023=0.6098  2024=0.6176
현행을 두 해 모두 이기는 조합 없음 → v7 유지
{"theta": 0.3, "stuck_w": 0.5, "blend": 0.7}
